# MNIST Classification with PyTorch + Weights & Biases

This notebook is a Jupyter-notebook version of the standard PyTorch MNIST example
(`mnist_classify.py`), with [Weights & Biases](https://wandb.ai/site/) (`wandb`) added
for experiment tracking. Instead of reading loss values from console output, every
training run is logged to a `wandb` dashboard where you can compare hyperparameters,
watch live loss/accuracy curves, and inspect the model's gradients and weights.

**What's new compared to the script:**
- `wandb.login()` / `wandb.init()` to start a tracked run
- `wandb.watch()` on the model to track gradients and weights
- `wandb.log()` calls inside the training and test loops
- `wandb.finish()` at the end to close out the run


In [ ]:
from __future__ import print_function
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

import wandb

## Log in to Weights & Biases

Running this once per environment authenticates the notebook with your `wandb` account (create a free one at [wandb.ai](https://wandb.ai/site/) if needed).

In [ ]:
wandb.login()

## Model definition

Same CNN architecture as the original script: two convolutional layers, two dropout layers, and two fully connected layers.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # First convolutional layer:
        # - Input channels: 1 (e.g., grayscale images)
        # - Output channels: 32
        # - Kernel size: 3x3
        # - Stride: 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1)

        # Second convolutional layer:
        # - Input channels: 32 (from conv1)
        # - Output channels: 64
        # - Kernel size: 3x3
        # - Stride: 1
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1)

        # First dropout layer:
        # - Dropout probability: 25%
        # - Helps prevent overfitting by randomly zeroing some of the elements of the input tensor
        self.dropout1 = nn.Dropout(p=0.25)

        # Second dropout layer:
        # - Dropout probability: 50%
        # - Further regularizes the model during training
        self.dropout2 = nn.Dropout(p=0.5)

        # First fully connected (linear) layer:
        # - Input features: 9216
        #   (Assuming input image size is 28x28, after two conv layers and pooling)
        # - Output features: 128
        self.fc1 = nn.Linear(in_features=9216, out_features=128)

        # Second fully connected (linear) layer:
        # - Input features: 128
        # - Output features: 10 (e.g., number of classes for classification)
        self.fc2 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        # Pass input through the first convolutional layer
        x = self.conv1(x)
        # Apply ReLU activation function
        x = F.relu(x)

        # Pass the result through the second convolutional layer
        x = self.conv2(x)
        # Apply ReLU activation function
        x = F.relu(x)

        # Apply 2D max pooling with a kernel size of 2
        # This reduces the spatial dimensions by a factor of 2
        x = F.max_pool2d(x, kernel_size=2)

        # Apply the first dropout layer
        x = self.dropout1(x)

        # Flatten the tensor starting from the first dimension (excluding batch size)
        # This prepares the data for the fully connected layers
        x = torch.flatten(x, 1)

        # Pass through the first fully connected layer
        x = self.fc1(x)
        # Apply ReLU activation function
        x = F.relu(x)

        # Apply the second dropout layer
        x = self.dropout2(x)

        # Pass through the second fully connected layer
        x = self.fc2(x)

        # Apply log softmax activation function to obtain log probabilities for each class
        output = F.log_softmax(x, dim=1)

        return output

## Training and evaluation loops

Both functions now call `wandb.log()` so each metric shows up as a live chart on the run's dashboard, in addition to the usual console `print`.

In [ ]:
def train(config, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % config["log_interval"] == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            # Log the running training loss to wandb
            wandb.log({"epoch": epoch, "train_loss": loss.item()})
            if config["dry_run"]:
                break

In [ ]:
def test(model, device, test_loader, epoch):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100. * correct / len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset), test_accuracy))

    # Log test metrics to wandb
    wandb.log({"epoch": epoch, "test_loss": test_loss, "test_accuracy": test_accuracy})

## Configuration

The script's `argparse` arguments become a plain `config` dictionary, since a notebook
has no command line. This same dictionary is passed straight to `wandb.init()`, so every
hyperparameter is automatically recorded with the run and easy to compare across
experiments on the dashboard.

In [ ]:
config = {
    "batch_size": 64,
    "test_batch_size": 1000,
    "epochs": 14,
    "lr": 1.0,
    "gamma": 0.7,
    "no_cuda": False,
    "no_mps": False,
    "dry_run": False,
    "seed": 1,
    "log_interval": 10,
    "save_model": False,
}

## Initialize a wandb run

This starts a tracked run under the `mnist-cnn` project and uploads `config` as the run's hyperparameters.

In [ ]:
run = wandb.init(project="mnist-cnn", config=config)

## Set up device, data, and model

`wandb.watch()` attaches to the model so gradient and weight histograms are logged automatically during training.

In [ ]:
use_cuda = not config["no_cuda"] and torch.cuda.is_available()
use_mps = not config["no_mps"] and torch.backends.mps.is_available()

torch.manual_seed(config["seed"])

if use_cuda:
    device = torch.device("cuda")
elif use_mps:
    device = torch.device("mps")
else:
    device = torch.device("cpu")

train_kwargs = {'batch_size': config["batch_size"]}
test_kwargs = {'batch_size': config["test_batch_size"]}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                   'pin_memory': True,
                   'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset1 = datasets.MNIST('data/', train=True, download=True,
                   transform=transform)
dataset2 = datasets.MNIST('data/', train=False,
                   transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1, **train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=config["lr"])
scheduler = StepLR(optimizer, step_size=1, gamma=config["gamma"])

# Track gradients and weight histograms for this model
wandb.watch(model, log="all", log_freq=100)

## Run training

Same epoch loop as the script — `train()` and `test()` now stream their metrics to the wandb dashboard as they run.

In [ ]:
for epoch in range(1, config["epochs"] + 1):
    train(config, model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader, epoch)
    scheduler.step()

if config["save_model"]:
    torch.save(model.state_dict(), "mnist_cnn.pt")
    wandb.save("mnist_cnn.pt")

# Close out the run and sync any remaining data
wandb.finish()